In [1]:
"""
================================================================================
  13_WILLIE_CrossModel_Comparison.ipynb
  MINI vs BASE vs XL - Scaling Analysis & Publication Visuals
================================================================================

  PRODUCES (saved to FIGURES_DIR, 300 DPI, PNG + PDF):
    1. fig_task_comparison        - 3 tasks x 3 models grouped bars
    2. fig_radar_overlay          - All 3 models on one radar chart
    3. fig_perclass_heatmap       - Per-class accuracy heatmap
    4. fig_scaling_curve          - Performance vs parameters
    5. fig_seg_comparison         - Segmentation detail
    6. fig_det_comparison         - Detection detail
    7. fig_combined_scores        - Combined multi-task scores
    8. fig_confusion_side_by_side - 3 confusion matrices
    9. fig_architecture_overview  - Params + efficiency + time
   10. tbl_master_comparison      - Full publication table
================================================================================
"""

import os, warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

PROJECT_ROOT = "."
FIGURES_DIR = os.path.join(PROJECT_ROOT, "artifacts/13_cross_model_comparison/figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

CLASS_NAMES = ["Diabetic", "Pressure", "Surgical", "Venous", "No Wound"]
CLASS_SHORT = ["DIA", "PRS", "SUR", "VEN", "NW"]
N_CLS = 5
MODEL_COLORS = {'MINI': '#3498db', 'BASE': '#f39c12', 'XL': '#e74c3c'}
MODEL_LIST = ['MINI', 'BASE', 'XL']

plt.rcParams.update({
    'font.size': 11, 'axes.titlesize': 14, 'axes.labelsize': 12,
    'figure.dpi': 100, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'savefig.facecolor': 'white', 'axes.grid': True, 'grid.alpha': 0.3,
    'font.family': 'sans-serif',
})

def save_fig(fig, name, close=True):
    png = os.path.join(FIGURES_DIR, f"{name}.png")
    pdf = os.path.join(FIGURES_DIR, f"{name}.pdf")
    fig.savefig(png, dpi=300, bbox_inches='tight', facecolor='white')
    fig.savefig(pdf, bbox_inches='tight', facecolor='white')
    if close:
        plt.close(fig)
    print(f"  📈 {name} (.png + .pdf)")

print("=" * 80)
print("  WILLIE Cross-Model Comparison: MINI vs BASE vs XL")
print("=" * 80)
print(f"  📁 Figures: {FIGURES_DIR}")

# ====================================================================
# RESULTS DATA (from notebooks 09, 10, 11)
# ====================================================================

MINI = {
    'name': 'MINI', 'notebook': '09', 'backbones': 'DINOv2-S',
    'params_total': 34.32, 'params_trainable': 12.26, 'params_frozen': 22.06,
    'cls_acc': 86.8, 'cls_f1': 85.4, 'cls_auc': 0.0,
    'seg_dice': 84.1, 'det_ap50': 86.7,
    'combined_3way': (86.8 + 84.1 + 86.7) / 3,
    'per_class_acc': {
        'Diabetic': 71.7, 'Pressure': 85.7, 'Surgical': 93.5,
        'Venous': 100.0, 'No Wound': 76.5,
    },
    'per_class_f1': {
        'Diabetic': 81.5, 'Pressure': 84.7, 'Surgical': 91.3,
        'Venous': 100.0, 'No Wound': 69.3,
    },
    'confusion_matrix': np.array([
        [33, 0, 2, 0, 11],
        [0, 36, 1, 1,  4],
        [0,  1, 58, 0,  3],
        [0,  0,  0, 50,  0],
        [2,  6,  0,  0, 26],
    ]),
    'train_time_hrs': 8.0,
    'fold_accs': [86.1, 87.5, 84.3, 86.6, 87.5],
    'fold_dices': [82.8, 85.3, 83.5, 85.3, 80.9],
    'fold_det': [84.8, 85.6, 84.4, 89.9, 82.1],
}

BASE = {
    'name': 'BASE', 'notebook': '10', 'backbones': 'DINOv2-L + ConvNeXt-L',
    'params_total': 520.4, 'params_trainable': 19.8, 'params_frozen': 500.6,
    'cls_acc': 91.88, 'cls_f1': 91.14, 'cls_auc': 0.9917,
    'seg_dice': 86.36, 'det_ap50': 89.91,
    'combined_3way': (91.88 + 86.36 + 89.91) / 3,
    'per_class_acc': {
        'Diabetic': 78.26, 'Pressure': 88.24, 'Surgical': 90.48,
        'Venous': 98.39, 'No Wound': 100.0,
    },
    'per_class_f1': {
        'Diabetic': 87.80, 'Pressure': 83.33, 'Surgical': 92.68,
        'Venous': 93.85, 'No Wound': 98.04,
    },
    'confusion_matrix': np.array([
        [36, 0, 2, 1, 7],
        [0, 30, 0, 2, 2],
        [0,  2, 38, 2, 0],
        [0,  0,  0, 61, 1],
        [0,  0,  0,  0, 50],
    ]),
    'train_time_hrs': 40.0,
    'fold_accs': [86.57, 88.0, 85.0, 90.0, 85.0],
    'fold_dices': [87.77, 89.0, 85.7, 87.0, 85.0],
    'fold_det': [89.49, 89.20, 80.71, 89.84, 80.18],
}

XL = {
    'name': 'XL', 'notebook': '11',
    'backbones': 'DINOv2-L + ConvNeXt-L + SAM2-Hiera-L',
    'params_total': 762.5, 'params_trainable': 360.2, 'params_frozen': 402.3,
    'cls_acc': 91.88, 'cls_f1': 90.73, 'cls_auc': 0.9856,
    'seg_dice': 91.41, 'det_ap50': 96.23,
    'combined_3way': 93.17,
    'per_class_acc': {
        'Diabetic': 82.6, 'Pressure': 92.9, 'Surgical': 100.0,
        'Venous': 98.0, 'No Wound': 79.4,
    },
    'per_class_f1': {
        'Diabetic': 89.4, 'Pressure': 90.7, 'Surgical': 96.1,
        'Venous': 98.0, 'No Wound': 79.4,
    },
    'confusion_matrix': np.array([
        [38, 1, 2, 0, 5],
        [0, 39, 0, 1, 2],
        [0,  0, 62, 0, 0],
        [0,  0,  1, 49, 0],
        [1,  4,  2,  0, 27],
    ]),
    'train_time_hrs': 77.0,
    'fold_accs': [89.8, 91.2, 86.6, 86.3, 84.6],
    'fold_dices': [92.2, 92.3, 91.5, 90.3, 90.9],
    'fold_det': [0, 0, 0, 0, 0],
}

MODELS = [MINI, BASE, XL]

print()
print(f"  {'Model':<8s} {'Params':>8s} {'Cls Acc':>8s} {'Seg Dice':>9s} {'Det AP50':>9s} {'Combined':>9s}")
print(f"  {'_'*55}")
for m in MODELS:
    print(f"  {m['name']:<8s} {m['params_total']:>7.1f}M "
          f"{m['cls_acc']:>7.2f}% {m['seg_dice']:>8.2f}% "
          f"{m['det_ap50']:>8.2f}% {m['combined_3way']:>8.2f}%")


# ====================================================================
# FIGURE 1: 3 TASKS x 3 MODELS GROUPED BAR CHART
# ====================================================================
print("\n" + "=" * 70)
print("  FIGURE 1: Task Comparison")
print("=" * 70)

fig1, ax = plt.subplots(figsize=(12, 6))
tasks = ['Classification\n(Accuracy %)', 'Segmentation\n(Dice %)', 'Detection\n(AP@0.5 %)']
x = np.arange(len(tasks))
w = 0.22

for i, m in enumerate(MODELS):
    vals = [m['cls_acc'], m['seg_dice'], m['det_ap50']]
    bars = ax.bar(x + (i - 1) * w, vals, w, label=m['name'],
                  color=MODEL_COLORS[m['name']], edgecolor='white', alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(tasks, fontweight='bold', fontsize=12)
ax.set_ylabel('Score (%)', fontweight='bold', fontsize=12)
ax.set_title('willie Multi-Task Performance Across Variants\n'
             'MINI (34.3M) vs BASE (520.4M) vs XL (762.5M)',
             fontweight='bold', fontsize=14, pad=15)
ax.legend(fontsize=11, loc='lower right')
ax.set_ylim([75, 102])
save_fig(fig1, "fig_task_comparison")


# ====================================================================
# FIGURE 2: RADAR CHART OVERLAY
# ====================================================================
print("\n" + "=" * 70)
print("  FIGURE 2: Radar Chart Overlay")
print("=" * 70)

fig2, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
categories = ['Cls Acc', 'Cls F1', 'Seg Dice', 'Det AP@0.5', 'Combined']
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

for m in MODELS:
    vals = [m['cls_acc'], m['cls_f1'], m['seg_dice'], m['det_ap50'], m['combined_3way']]
    vals += vals[:1]
    ax.plot(angles, vals, 'o-', linewidth=2.5, markersize=7,
            color=MODEL_COLORS[m['name']], label=m['name'])
    ax.fill(angles, vals, alpha=0.08, color=MODEL_COLORS[m['name']])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontweight='bold', fontsize=11)
ax.set_ylim(75, 100)
ax.set_yticks([80, 85, 90, 95, 100])
ax.set_yticklabels(['80%', '85%', '90%', '95%', '100%'], fontsize=8)
ax.legend(loc='upper right', bbox_to_anchor=(1.25, 1.1), fontsize=11, framealpha=0.9)
ax.set_title('willie Multi-Task Profile\nMINI vs BASE vs XL',
             fontweight='bold', fontsize=14, pad=30)
save_fig(fig2, "fig_radar_overlay")


# ====================================================================
# FIGURE 3: PER-CLASS ACCURACY HEATMAP
# ====================================================================
print("\n" + "=" * 70)
print("  FIGURE 3: Per-Class Heatmap")
print("=" * 70)

fig3, ax = plt.subplots(figsize=(10, 4.5))
heatmap_data = np.array([
    [m['per_class_acc'][c] for c in CLASS_NAMES] for m in MODELS
])
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='RdYlGn',
            xticklabels=CLASS_NAMES, yticklabels=MODEL_LIST,
            linewidths=0.5, linecolor='white', vmin=60, vmax=100,
            cbar_kws={'label': 'Accuracy (%)', 'shrink': 0.8},
            annot_kws={'fontsize': 13, 'fontweight': 'bold'}, ax=ax)
ax.set_xlabel('Wound Class', fontweight='bold', fontsize=12)
ax.set_ylabel('Model Variant', fontweight='bold', fontsize=12)
ax.set_title('Per-Class Classification Accuracy (%)',
             fontweight='bold', fontsize=14, pad=15)
save_fig(fig3, "fig_perclass_heatmap")


# ====================================================================
# FIGURE 4: SCALING CURVE
# ====================================================================
print("\n" + "=" * 70)
print("  FIGURE 4: Scaling Curve")
print("=" * 70)

fig4, axes = plt.subplots(1, 3, figsize=(16, 5))
params = [m['params_total'] for m in MODELS]
task_data = [
    ('Classification (Acc %)', [m['cls_acc'] for m in MODELS]),
    ('Segmentation (Dice %)', [m['seg_dice'] for m in MODELS]),
    ('Detection (AP@0.5 %)', [m['det_ap50'] for m in MODELS]),
]
for ax, (title, vals) in zip(axes, task_data):
    for i, m in enumerate(MODELS):
        ax.scatter(params[i], vals[i], s=200, c=MODEL_COLORS[m['name']],
                   edgecolors='black', linewidths=1.5, zorder=3)
        ax.annotate(f"{m['name']}\n{vals[i]:.1f}%",
                    xy=(params[i], vals[i]),
                    xytext=(params[i], vals[i] + 1.5),
                    ha='center', fontsize=10, fontweight='bold')
    ax.plot(params, vals, '--', color='gray', alpha=0.5, lw=1.5)
    ax.set_xlabel('Total Parameters (M)', fontweight='bold')
    ax.set_ylabel(title.split('(')[0].strip(), fontweight='bold')
    ax.set_title(title, fontweight='bold')
fig4.suptitle('Performance Scaling with Model Size',
              fontweight='bold', fontsize=14, y=1.03)
plt.tight_layout()
save_fig(fig4, "fig_scaling_curve")


# ====================================================================
# FIGURE 5: SEGMENTATION DETAIL
# ====================================================================
print("\n" + "=" * 70)
print("  FIGURE 5: Segmentation Comparison")
print("=" * 70)

fig5, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for i, m in enumerate(MODELS):
    fd = m['fold_dices']
    if any(d > 0 for d in fd):
        ax1.plot(range(5), fd, 'o-', color=MODEL_COLORS[m['name']],
                 lw=2, markersize=8,
                 label=f"{m['name']} (mean={np.mean(fd):.1f}%)")
        ax1.axhline(np.mean(fd), color=MODEL_COLORS[m['name']],
                     linestyle=':', alpha=0.4)
ax1.set_xticks(range(5))
ax1.set_xticklabels([f'Fold {i}' for i in range(5)], fontweight='bold')
ax1.set_ylabel('Dice Score (%)', fontweight='bold')
ax1.set_title('Segmentation Dice per Fold', fontweight='bold')
ax1.legend(fontsize=9)

dice_vals = [m['seg_dice'] for m in MODELS]
bars = ax2.bar(MODEL_LIST, dice_vals,
               color=[MODEL_COLORS[n] for n in MODEL_LIST],
               edgecolor='black', linewidth=0.5, alpha=0.85, width=0.5)
for bar, val in zip(bars, dice_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.2f}%', ha='center', fontweight='bold', fontsize=12)
ax2.set_ylabel('Mean Dice (%)', fontweight='bold')
ax2.set_title('Mean Segmentation Dice', fontweight='bold')
ax2.set_ylim([78, 96])

fig5.suptitle('WILLIE Segmentation Performance',
              fontweight='bold', fontsize=14, y=1.03)
plt.tight_layout()
save_fig(fig5, "fig_seg_comparison")


# ====================================================================
# FIGURE 6: DETECTION DETAIL
# ====================================================================
print("\n" + "=" * 70)
print("  FIGURE 6: Detection Comparison")
print("=" * 70)

fig6, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ap_vals = [m['det_ap50'] for m in MODELS]
bars = ax1.bar(MODEL_LIST, ap_vals,
               color=[MODEL_COLORS[n] for n in MODEL_LIST],
               edgecolor='black', linewidth=0.5, alpha=0.85, width=0.5)
for bar, val in zip(bars, ap_vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.2f}%', ha='center', fontweight='bold', fontsize=12)
ax1.set_ylabel('AP@0.5 (%)', fontweight='bold')
ax1.set_title('Detection AP@0.5 (Seg-to-Det)', fontweight='bold')
ax1.set_ylim([80, 102])

improvements = [0, ap_vals[1] - ap_vals[0], ap_vals[2] - ap_vals[0]]
bars2 = ax2.bar(MODEL_LIST, improvements,
                color=[MODEL_COLORS[n] for n in MODEL_LIST],
                edgecolor='black', linewidth=0.5, alpha=0.85, width=0.5)
for bar, val in zip(bars2, improvements):
    ax2.text(bar.get_x() + bar.get_width()/2, max(val + 0.2, 0.2),
             f'+{val:.1f}%', ha='center', fontweight='bold', fontsize=11)
ax2.set_ylabel('Improvement over MINI (%)', fontweight='bold')
ax2.set_title('Detection Gain vs MINI', fontweight='bold')

fig6.suptitle('WILLIE Detection Performance (Seg-to-Det)',
              fontweight='bold', fontsize=14, y=1.03)
plt.tight_layout()
save_fig(fig6, "fig_det_comparison")


# ====================================================================
# FIGURE 7: COMBINED MULTI-TASK SCORES
# ====================================================================
print("\n" + "=" * 70)
print("  FIGURE 7: Combined Scores")
print("=" * 70)

fig7, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))

cls_vals = [m['cls_acc'] for m in MODELS]
seg_vals = [m['seg_dice'] for m in MODELS]
det_vals = [m['det_ap50'] for m in MODELS]
x = np.arange(3)
w = 0.5

ax1.bar(x, [v/3 for v in cls_vals], w,
        label='Classification', color='#3498db', alpha=0.85)
ax1.bar(x, [v/3 for v in seg_vals], w,
        bottom=[v/3 for v in cls_vals],
        label='Segmentation', color='#2ecc71', alpha=0.85)
ax1.bar(x, [v/3 for v in det_vals], w,
        bottom=[(c+s)/3 for c, s in zip(cls_vals, seg_vals)],
        label='Detection', color='#e74c3c', alpha=0.85)
for i, m in enumerate(MODELS):
    ax1.text(i, m['combined_3way'] + 0.5,
             f"{m['combined_3way']:.1f}%",
             ha='center', fontweight='bold', fontsize=12)
ax1.set_xticks(x)
ax1.set_xticklabels(MODEL_LIST, fontweight='bold', fontsize=12)
ax1.set_ylabel('Score Contribution (%)', fontweight='bold')
ax1.set_title('Combined Score Breakdown\n(Cls + Seg + Det) / 3', fontweight='bold')
ax1.legend(fontsize=9)
ax1.set_ylim([0, 100])

combined_vals = [m['combined_3way'] for m in MODELS]
bars = ax2.bar(MODEL_LIST, combined_vals,
               color=[MODEL_COLORS[n] for n in MODEL_LIST],
               edgecolor='black', linewidth=0.5, alpha=0.85, width=0.5)
for bar, val in zip(bars, combined_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.2f}%', ha='center', fontweight='bold', fontsize=13)
ax2.axhline(95, color='red', linestyle='--', lw=2, alpha=0.5,
            label='Target: 95%')
ax2.set_ylabel('Combined Score (%)', fontweight='bold')
ax2.set_title('Combined Multi-Task Score\nvs 95% Target', fontweight='bold')
ax2.legend(fontsize=10)
ax2.set_ylim([80, 100])

fig7.suptitle('WILLIE Multi-Task Combined Performance',
              fontweight='bold', fontsize=14, y=1.03)
plt.tight_layout()
save_fig(fig7, "fig_combined_scores")


# ====================================================================
# FIGURE 8: CONFUSION MATRICES SIDE-BY-SIDE
# ====================================================================
print("\n" + "=" * 70)
print("  FIGURE 8: Confusion Matrices")
print("=" * 70)

fig8, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, m in zip(axes, MODELS):
    cm = m['confusion_matrix']
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt='.0%', cmap='Blues',
                xticklabels=CLASS_SHORT, yticklabels=CLASS_SHORT,
                linewidths=0.5, linecolor='white', vmin=0, vmax=1,
                ax=ax, cbar=False, annot_kws={'fontsize': 11})
    acc = np.trace(cm) / cm.sum()
    ax.set_title(f"{m['name']} ({m['params_total']:.0f}M)\n"
                 f"Acc: {acc:.1%}", fontweight='bold', fontsize=12)
    ax.set_xlabel('Predicted')
    if m == MODELS[0]:
        ax.set_ylabel('True')

fig8.suptitle('WILLIE Classification Confusion Matrices '
              '(Test Set, n=234)',
              fontweight='bold', fontsize=14, y=1.05)
plt.tight_layout()
save_fig(fig8, "fig_confusion_side_by_side")


# ====================================================================
# FIGURE 9: ARCHITECTURE OVERVIEW
# ====================================================================
print("\n" + "=" * 70)
print("  FIGURE 9: Architecture Overview")
print("=" * 70)

fig9, axes = plt.subplots(1, 3, figsize=(15, 5))

x_pos = np.arange(3)
total = [m['params_total'] for m in MODELS]
trainable = [m['params_trainable'] for m in MODELS]
frozen = [m['params_frozen'] for m in MODELS]

axes[0].bar(x_pos, trainable, 0.3, label='Trainable',
            color='#e74c3c', alpha=0.85)
axes[0].bar(x_pos, frozen, 0.3, bottom=trainable,
            label='Frozen', color='#95a5a6', alpha=0.6)
for i, (t, tr) in enumerate(zip(total, trainable)):
    axes[0].text(i, t + 10,
                 f'{t:.0f}M\n({tr:.0f}M train)',
                 ha='center', fontsize=9, fontweight='bold')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(MODEL_LIST, fontweight='bold')
axes[0].set_ylabel('Parameters (M)', fontweight='bold')
axes[0].set_title('Model Size Breakdown', fontweight='bold')
axes[0].legend(fontsize=9)

efficiency = [m['combined_3way'] / m['params_total'] for m in MODELS]
bars = axes[1].bar(MODEL_LIST, efficiency,
                   color=[MODEL_COLORS[n] for n in MODEL_LIST],
                   edgecolor='black', linewidth=0.5, alpha=0.85, width=0.5)
for bar, val in zip(bars, efficiency):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', fontweight='bold', fontsize=11)
axes[1].set_ylabel('Combined Score / M params', fontweight='bold')
axes[1].set_title('Parameter Efficiency\n(higher = more efficient)',
                   fontweight='bold')

times = [m['train_time_hrs'] for m in MODELS]
bars = axes[2].bar(MODEL_LIST, times,
                   color=[MODEL_COLORS[n] for n in MODEL_LIST],
                   edgecolor='black', linewidth=0.5, alpha=0.85, width=0.5)
for bar, val in zip(bars, times):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.0f}h', ha='center', fontweight='bold', fontsize=12)
axes[2].set_ylabel('Training Time (hours)', fontweight='bold')
axes[2].set_title('5-Fold CV Training Time', fontweight='bold')

fig9.suptitle('WILLIE Architecture Complexity & Efficiency',
              fontweight='bold', fontsize=14, y=1.03)
plt.tight_layout()
save_fig(fig9, "fig_architecture_overview")


# ====================================================================
# FIGURE 10: MASTER TABLE
# ====================================================================
print("\n" + "=" * 70)
print("  FIGURE 10: Master Comparison Table")
print("=" * 70)

fig10, ax = plt.subplots(figsize=(16, 7))
ax.axis('off')

table_data = [
    ['', 'MINI', 'BASE', 'XL'],
    ['Notebook', '09', '10', '11'],
    ['Backbones', 'DINOv2-S',
     'DINOv2-L +\nConvNeXt-L',
     'DINOv2-L + ConvNeXt-L\n+ SAM2-Hiera-L'],
    ['Total Params', '34.3M', '520.4M', '762.5M'],
    ['Trainable', '12.3M', '19.8M', '360.2M'],
    ['Cls Accuracy', '86.80%', '91.88%', '91.88%'],
    ['Cls F1 (macro)', '85.40%', '91.14%', '90.73%'],
    ['Seg Dice', '84.10%', '86.36%', '91.41%'],
    ['Det AP@0.5', '86.70%', '89.91%', '96.23%'],
    ['Combined', '85.87%', '89.38%', '93.17%'],
    ['Train Time', '~8h', '~40h', '~77h'],
    ['Best For', 'Speed /\nDeployment', 'Classification',
     'Overall Best /\nAll Tasks'],
]

table = ax.table(
    cellText=table_data[1:], colLabels=table_data[0],
    cellLoc='center', loc='center',
    colWidths=[0.16, 0.16, 0.20, 0.28])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.7)

for j in range(4):
    table[0, j].set_facecolor('#2c3e50')
    table[0, j].set_text_props(color='white', fontweight='bold', fontsize=12)

col_tints = ['#ffffff', '#ebf5fb', '#fef9e7', '#fdedec']
for i in range(1, len(table_data)):
    for j in range(4):
        table[i, j].set_facecolor(col_tints[j])

# Highlight combined row
for j in range(4):
    table[9, j].set_facecolor('#d5f5e3')
    table[9, j].set_text_props(fontweight='bold')

ax.set_title('willie Complete Cross-Model Comparison\n'
             'All models: 5-fold CV on WILLIE dataset '
             '(FUSeg + AZH + Medetec)',
             fontweight='bold', fontsize=14, pad=25)
save_fig(fig10, "tbl_master_comparison")


# ====================================================================
# SUMMARY
# ====================================================================

fig_files = sorted([f for f in os.listdir(FIGURES_DIR)
                    if f.endswith(('.png', '.pdf'))])

print("\n" + "=" * 80)
print("  CROSS-MODEL COMPARISON COMPLETE")
print("=" * 80)

print(f"""
  SCALING STORY:
  Variant  | Params  | Cls Acc | Seg Dice | Det AP50 | Combined
  MINI     |  34.3M  |  86.80% |  84.10%  |  86.70%  |  85.87%
  BASE     | 520.4M  |  91.88% |  86.36%  |  89.91%  |  89.38%
  XL       | 762.5M  |  91.88% |  91.41%  |  96.23%  |  93.17%

  KEY FINDINGS:
  - XL dominates segmentation (+7.3% over MINI, +5.1% over BASE)
  - XL dominates detection (+9.5% over MINI, +6.3% over BASE)
  - Classification saturates at ~92% for both BASE and XL
  - MINI is 22x smaller but only 7.3% behind XL overall
  - Combined: MINI 85.9% -> BASE 89.4% -> XL 93.2%

  {FIGURES_DIR}
  {len(fig_files)} files generated
""")

for f in fig_files:
    size = os.path.getsize(os.path.join(FIGURES_DIR, f))
    icon = '📈' if f.endswith('.png') else '📄'
    print(f"     {icon} {f}  ({size/1024:.1f} KB)")

print("""
  ┌────────────────────────────────────────────────────────────┐
  │  FIGURE INVENTORY                                          │
  ├────────────────────────────────────────────────────────────┤
  │  1. fig_task_comparison       - 3 tasks x 3 models bars    │
  │  2. fig_radar_overlay         - All 3 models radar chart   │
  │  3. fig_perclass_heatmap      - 5 classes x 3 models       │
  │  4. fig_scaling_curve         - Performance vs parameters   │
  │  5. fig_seg_comparison        - Segmentation detail         │
  │  6. fig_det_comparison        - Detection detail            │
  │  7. fig_combined_scores       - Combined score breakdown    │
  │  8. fig_confusion_side_by_side- 3 confusion matrices        │
  │  9. fig_architecture_overview - Params + efficiency + time  │
  │ 10. tbl_master_comparison     - Full publication table      │
  └────────────────────────────────────────────────────────────┘
""")

  WILLIE Cross-Model Comparison: MINI vs BASE vs XL
  📁 Figures: artifacts/13_cross_model_comparison/figures

  Model      Params  Cls Acc  Seg Dice  Det AP50  Combined
  _______________________________________________________
  MINI        34.3M   86.80%    84.10%    86.70%    85.87%
  BASE       520.4M   91.88%    86.36%    89.91%    89.38%
  XL         762.5M   91.88%    91.41%    96.23%    93.17%

  FIGURE 1: Task Comparison
  📈 fig_task_comparison (.png + .pdf)

  FIGURE 2: Radar Chart Overlay
  📈 fig_radar_overlay (.png + .pdf)

  FIGURE 3: Per-Class Heatmap
  📈 fig_perclass_heatmap (.png + .pdf)

  FIGURE 4: Scaling Curve
  📈 fig_scaling_curve (.png + .pdf)

  FIGURE 5: Segmentation Comparison
  📈 fig_seg_comparison (.png + .pdf)

  FIGURE 6: Detection Comparison
  📈 fig_det_comparison (.png + .pdf)

  FIGURE 7: Combined Scores
  📈 fig_combined_scores (.png + .pdf)

  FIGURE 8: Confusion Matrices
  📈 fig_confusion_side_by_side (.png + .pdf)

  FIGURE 9: Architecture Overview
  📈